# IFN680 Project 8 - Task 3: Comparative Analysis (Tiny NeRF vs Extended NeRF)
This notebook reproduces every number and figure in the report. It **does not train**: it loads
`weights/tiny_nerf.pth` and `weights/extended_nerf.pth` (plus the training histories in `results/`) and evaluates
both models on the held-out test views (images 100+ of `tiny_nerf_data.npz`, never used in training).

Contents
1. Setup, data and model definitions (identical to the training notebooks)
2. Quantitative: PSNR / SSIM per test view, coarse vs fine, inference time, parameter count
3. Training curves
4. Qualitative: test-view mosaic + error maps, depth maps, novel views
5. Hierarchical sampling: where the samples go along a ray
6. View-dependent appearance

Figures are also written to `results/figures/` for the report.

## 1. Setup

In [ ]:
import os
import json
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
torch.manual_seed(0); np.random.seed(0)

FIG_DIR = os.path.join('results', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
def savefig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name), dpi=200, bbox_inches='tight')

In [ ]:
data = np.load('tiny_nerf_data.npz')
images = torch.from_numpy(data['images']).float().to(device)
poses = torch.from_numpy(data['poses']).float().to(device)
focal = float(data['focal'])
H, W = images.shape[1:3]
K = torch.Tensor([[focal, 0, W/2.0], [0, focal, H/2.0], [0, 0, 1]]).float().to(device)

# same split as the training notebooks
test_images, test_poses = images[100:], poses[100:]
train_images, train_poses = images[:100], poses[:100]
print(f"train: {tuple(train_images.shape)}  test: {tuple(test_images.shape)}  focal={focal:.2f}")

In [ ]:
def get_rays(K, c2w, H, W):
    """
    TODO: Get ray origin, direction to each image pixels based on camera pose
    """
    # 1. Create pixel grid, x first
    i, j = torch.meshgrid(
        torch.arange(W, dtype=torch.float32, device=c2w.device),
        torch.arange(H, dtype=torch.float32, device=c2w.device),
        indexing='xy'
    )

    # 2. Get directions in camera coordinates
    # We use the explicit K components: (u - cx)/fx, (v - cy)/fy
    # NeRF typically uses -1 for the z-axis (looking "into" the screen)
    dirs = torch.stack([
        (i - K[0, 2]) / K[0, 0],
        -(j - K[1, 2]) / K[1, 1], # Negated to flip y-axis for standard NeRF orientation
        -torch.ones_like(i)
    ], dim=-1)

    # 3. Transform directions to world space
    # (H, W, 3) @ (3, 3) -> (H, W, 3)
    rays_d = dirs @ c2w[:3, :3].T
    
    # 4. Origin is the translation vector of the camera-to-world matrix
    rays_o = c2w[:3, -1].expand(rays_d.shape)
    return rays_o, rays_d

### Model definitions (copied from `TinyNeRF.ipynb` and `ExtendedNeRF.ipynb`)

In [ ]:
class PositionalEncoding(torch.nn.Module):
    """
    Positional encoding module
    """
    def __init__(self, L_embed=6):
        super(PositionalEncoding, self).__init__()
        # frequency bands used
        self.L_embed = L_embed
        self.bands = (2.0 ** torch.arange(L_embed, dtype=torch.float32).to(device)) * torch.pi
        self.bands = self.bands.view(1, L_embed, 1)  # shape (1, L_embed, 1)

    def forward(self, x):
        x_scaled = x.unsqueeze(-2) * self.bands  # shape (N, L, D)
        sin_cos = torch.cat([torch.sin(x_scaled), torch.cos(x_scaled)], dim=-2) # shape (N, 2*L, D)
        sin_cos = sin_cos.flatten(start_dim=-2)  # (N, 2*L*D)
        encoded = torch.cat([x, sin_cos], dim=-1)  # (N, D + 2*L*D)
        return encoded


class TinyNeRF(torch.nn.Module):
    """
    Tiny NeRF baseline (Tutorial 10.4): a single small MLP that maps the encoded 3D
    position gamma(x) to (sigma, rgb). The viewing direction is ignored -> matte renderer.
    The `dir` argument is accepted only so both models share the same interface.
    """
    def __init__(self, filter_size=128, L_embed=6):
        super(TinyNeRF, self).__init__()
        self.positional_encoding = PositionalEncoding(L_embed)
        x_dim = 3 + 3 * 2 * L_embed
        self.layer1 = torch.nn.Linear(x_dim, filter_size)
        self.layer2 = torch.nn.Linear(filter_size, filter_size)
        self.layer3 = torch.nn.Linear(filter_size, filter_size)
        self.output_layer = torch.nn.Linear(filter_size, 4)

    def forward(self, x, dir=None):
        h = F.relu(self.layer1(self.positional_encoding(x)))
        h = F.relu(self.layer2(h))
        h = F.relu(self.layer3(h))
        out = self.output_layer(h)
        sigma = F.relu(out[..., :1])          # density >= 0
        rgb = torch.sigmoid(out[..., 1:])     # colour in [0, 1]
        return torch.cat([sigma, rgb], dim=-1)


class NeRF(torch.nn.Module):
    """
    NeRF MLP with view-direction dependency.
    Density depends on position only; color depends on position features + view direction.
    """
    def __init__(self, filter_size=128, L_embed=6, L_dir=4):
        super(NeRF, self).__init__()
        self.positional_encoding = PositionalEncoding(L_embed)
        self.direction_encoding = PositionalEncoding(L_dir)

        x_dim = 3 + 3 * 2 * L_embed
        d_dim = 3 + 3 * 2 * L_dir

        # Position trunk
        self.layer1 = torch.nn.Linear(x_dim, filter_size)
        self.layer2 = torch.nn.Linear(filter_size, filter_size)

        # Density
        self.sigma_layer = torch.nn.Linear(filter_size, 1)
        # geometric features
        self.feature_layer = torch.nn.Linear(filter_size, filter_size)

        # Color head conditioned on view direction
        self.color_layer1 = torch.nn.Linear(filter_size + d_dim, filter_size // 2)
        self.color_layer2 = torch.nn.Linear(filter_size // 2, 3)

    def forward(self, x, dir):
        # TASK 1: forward pass with view-direction conditioning
        # x:   [..., 3] sample positions,  dir: [..., 3] ray directions (same leading shape)

        # (1) Position trunk: gamma(x) -> h. Only the position is seen here, so density
        #     (geometry) cannot change with the camera angle -> multi-view consistency.
        h = F.relu(self.layer1(self.positional_encoding(x)))
        h = F.relu(self.layer2(h))

        # (2) Density head: sigma(x) >= 0 (ReLU), independent of the viewing direction
        sigma = F.relu(self.sigma_layer(h))                       # [..., 1]

        # (3) Geometric feature vector passed on to the colour branch
        #     (linear, no activation - as in Mildenhall et al.)
        feat = self.feature_layer(h)                              # [..., filter_size]

        # (4) Viewing direction: normalise to a unit vector and encode gamma(d)
        d = dir / torch.norm(dir, dim=-1, keepdim=True)
        d_enc = self.direction_encoding(d)                        # [..., d_dim]

        # (5) Colour head: c(x, d) in [0, 1] (sigmoid)
        c = F.relu(self.color_layer1(torch.cat([feat, d_enc], dim=-1)))
        rgb = torch.sigmoid(self.color_layer2(c))                 # [..., 3]

        return torch.cat([sigma, rgb], dim=-1)                    # [..., 4]

In [ ]:
def volume_rendering(sigma, rgb, t_vals):
    # Calculate distance between samples (delta)
    # t_vals shape: [N_rays, N_samples]
    deltas = t_vals[..., 1:] - t_vals[..., :-1]
    
    # Append a large value (1e10) for the last interval
    dist_shape = list(deltas.shape[:-1]) + [1]
    last_dist = torch.full(dist_shape, 1e10, device=sigma.device)
    deltas = torch.cat([deltas, last_dist], dim=-1)

    # alpha = 1.0 - exp(-sigma * delta)
    # sigma must be non-negative (usually ensured by ReLU in the MLP)
    alpha = 1.0 - torch.exp(-sigma * deltas)
    
    # Transmittance T_i = product of (1 - alpha_j) for j < i
    # We use cumprod on (1-alpha) shifted by one position
    # prepend 1.0 because T_1 is always 1.0
    shifted_transmittance = torch.cat([torch.ones_like(alpha[..., :1]), 1.0 - alpha + 1e-10], dim=-1)
    T = torch.cumprod(shifted_transmittance, dim=-1)[..., :-1]

    # color blending
    weights = alpha * T # [N_rays, N_samples]
    rgb_map = torch.sum(weights[..., None] * rgb, dim=-2) # [N_rays, 3]
    
    return rgb_map, weights


def sample_pdf(bins, weights, N_samples, rand=False):
    # Get PDF and CDF from weights
    weights = weights + 1e-5  # Prevent division by zero
    pdf = weights / torch.sum(weights, dim=-1, keepdim=True)
    cdf = torch.cumsum(pdf, dim=-1)
    cdf = torch.cat([torch.zeros_like(cdf[..., :1]), cdf], dim=-1) # [N_rays, N_bins + 1]

    # Sample uniform points u
    if rand:
        u = torch.rand(list(cdf.shape[:-1]) + [N_samples], device=cdf.device)
    else:
        u = torch.linspace(0., 1., steps=N_samples, device=cdf.device)
        u = u.expand(list(cdf.shape[:-1]) + [N_samples])

    u = u.contiguous()

    # Find indices where u fits in the CDF
    inds = torch.searchsorted(cdf, u, right=True)
    below = torch.max(torch.zeros_like(inds - 1), inds - 1)
    above = torch.min((cdf.shape[-1] - 1) * torch.ones_like(inds), inds)

    # Gather CDF values for the intervals
    # Input CDF: [N_rays, N_bins+1], Inds: [N_rays, N_samples]
    cdf_g0 = torch.gather(cdf, -1, below)
    cdf_g1 = torch.gather(cdf, -1, above)
    bins_g0 = torch.gather(bins, -1, below)
    bins_g1 = torch.gather(bins, -1, above)

    # Linear interpolation
    denom = (cdf_g1 - cdf_g0)
    denom = torch.where(denom < 1e-5, torch.ones_like(denom), denom)
    t = (u - cdf_g0) / denom
    samples = bins_g0 + t * (bins_g1 - bins_g0)

    return samples # [N_rays, N_samples]


def stratified_samples(N_rays, near, far, N_samples, rand, device):
    """Evenly spaced depths in [near, far]; with rand=True one jittered sample per bin."""
    t_vals = torch.linspace(near, far, N_samples, device=device).expand(N_rays, N_samples)
    if rand:
        # Add jitter: sample uniformly inside each bin [lower_i, upper_i]
        mids = 0.5 * (t_vals[..., 1:] + t_vals[..., :-1])
        upper = torch.cat([mids, t_vals[..., -1:]], dim=-1)
        lower = torch.cat([t_vals[..., :1], mids], dim=-1)
        t_vals = lower + (upper - lower) * torch.rand_like(t_vals)
    return t_vals                                                  # [N_rays, N_samples]


def query_network(network, rays_o, rays_d, t_vals):
    """Evaluate a radiance field at r(t) = o + t d for every depth in t_vals."""
    pts = rays_o[:, None, :] + rays_d[:, None, :] * t_vals[..., :, None]   # [N_rays, N_s, 3]
    dirs = rays_d[:, None, :].expand_as(pts)                                # [N_rays, N_s, 3]
    raw = network(pts, dirs)                                                # [N_rays, N_s, 4]
    return raw[..., 0], raw[..., 1:]                                        # sigma, rgb


def render_rays(coarse_network, fine_network, rays_o, rays_d, near, far, N_c, N_f, rand=False,
                return_extras=False):
    # TASK 2: ray rendering with hierarchical sampling (coarse + fine networks)
    # Accepts rays of any leading shape, e.g. [N_rays, 3] (training) or [H, W, 3] (full image)
    batch_shape = rays_o.shape[:-1]
    rays_o, rays_d = rays_o.reshape(-1, 3), rays_d.reshape(-1, 3)
    N_rays = rays_o.shape[0]

    # 1. Stratified Sampling (Coarse): t_vals [N_rays, N_c], jittered when rand=True
    t_vals = stratified_samples(N_rays, near, far, N_c, rand, rays_o.device)

    # Points in space r(t) = o + td  ->  Coarse pass
    sigma_c, rgb_c = query_network(coarse_network, rays_o, rays_d, t_vals)
    rgb_map_c, weights_c = volume_rendering(sigma_c, rgb_c, t_vals)

    # 2. Hierarchical Sampling (Fine)
    # The coarse weights w_i = T_i * alpha_i define a piecewise-constant PDF over the bins
    # whose edges are the midpoints between coarse samples. The first/last weights are
    # dropped because they have no bin on both sides (as in the original NeRF code).
    t_mids = 0.5 * (t_vals[..., 1:] + t_vals[..., :-1])            # [N_rays, N_c-1] bin edges
    t_fine = sample_pdf(t_mids, weights_c[..., 1:-1], N_f, rand=rand)
    t_fine = t_fine.detach()   # no gradient flows through the sample locations

    # Combine and sort all samples: N_c + N_f per ray
    t_all, _ = torch.sort(torch.cat([t_vals, t_fine], dim=-1), dim=-1)

    # Fine pass: fine network evaluated on all N_c + N_f points
    sigma_f, rgb_f = query_network(fine_network, rays_o, rays_d, t_all)
    rgb_map_f, weights_f = volume_rendering(sigma_f, rgb_f, t_all)

    rgb_map_c = rgb_map_c.reshape(*batch_shape, 3)
    rgb_map_f = rgb_map_f.reshape(*batch_shape, 3)

    if return_extras:
        extras = {
            't_coarse': t_vals, 'weights_coarse': weights_c,
            't_fine': t_all, 'weights_fine': weights_f,
            'depth_coarse': torch.sum(weights_c * t_vals, -1).reshape(batch_shape),
            'depth_fine': torch.sum(weights_f * t_all, -1).reshape(batch_shape),
            'acc_fine': torch.sum(weights_f, -1).reshape(batch_shape),
        }
        return rgb_map_c, rgb_map_f, extras
    return rgb_map_c, rgb_map_f


def render_rays_uniform(network, rays_o, rays_d, near, far, N_samples, rand=False, return_extras=False):
    """Tiny NeRF rendering: one network, stratified (uniform) samples only."""
    batch_shape = rays_o.shape[:-1]
    rays_o, rays_d = rays_o.reshape(-1, 3), rays_d.reshape(-1, 3)
    t_vals = stratified_samples(rays_o.shape[0], near, far, N_samples, rand, rays_o.device)
    sigma, rgb = query_network(network, rays_o, rays_d, t_vals)
    rgb_map, weights = volume_rendering(sigma, rgb, t_vals)
    rgb_map = rgb_map.reshape(*batch_shape, 3)
    if return_extras:
        extras = {'t': t_vals, 'weights': weights,
                  'depth': torch.sum(weights * t_vals, -1).reshape(batch_shape),
                  'acc': torch.sum(weights, -1).reshape(batch_shape)}
        return rgb_map, extras
    return rgb_map


@torch.no_grad()
def render_full(kind, rays_o, rays_d, chunk=1024, coarse=None, fine=None):
    """Render an [H, W] image in chunks. kind='tiny' or 'ext'. Returns rgb (and depth)."""
    H_, W_ = rays_o.shape[:2]
    ro, rd = rays_o.reshape(-1, 3), rays_d.reshape(-1, 3)
    coarse = model_coarse if coarse is None else coarse
    fine = model_fine if fine is None else fine
    out = {'rgb': [], 'depth': [], 'rgb_coarse': []}
    for i in range(0, ro.shape[0], chunk):
        if kind == 'tiny':
            rgb, ex = render_rays_uniform(model_tiny, ro[i:i+chunk], rd[i:i+chunk], NEAR, FAR,
                                          TINY_CFG['N_samples'], rand=False, return_extras=True)
            out['rgb'].append(rgb); out['depth'].append(ex['depth'])
        else:
            rgb_c, rgb_f, ex = render_rays(coarse, fine, ro[i:i+chunk], rd[i:i+chunk], NEAR, FAR,
                                           EXT_CFG['N_c'], EXT_CFG['N_f'], rand=False, return_extras=True)
            out['rgb'].append(rgb_f); out['rgb_coarse'].append(rgb_c); out['depth'].append(ex['depth_fine'])
    return {k: torch.cat(v).reshape(H_, W_, -1).squeeze(-1) for k, v in out.items() if len(v)}

### Load the trained weights

In [ ]:
ckpt_tiny = torch.load('weights/tiny_nerf.pth', map_location=device)
ckpt_ext = torch.load('weights/extended_nerf.pth', map_location=device)
TINY_CFG, EXT_CFG = ckpt_tiny['config'], ckpt_ext['config']
NEAR, FAR = EXT_CFG['near'], EXT_CFG['far']

model_tiny = TinyNeRF(TINY_CFG['filter_size'], TINY_CFG['L_embed']).to(device)
model_tiny.load_state_dict(ckpt_tiny['model'])
model_coarse = NeRF(EXT_CFG['filter_size'], EXT_CFG['L_embed'], EXT_CFG['L_dir']).to(device)
model_fine = NeRF(EXT_CFG['filter_size'], EXT_CFG['L_embed'], EXT_CFG['L_dir']).to(device)
model_coarse.load_state_dict(ckpt_ext['coarse']); model_fine.load_state_dict(ckpt_ext['fine'])
for m in (model_tiny, model_coarse, model_fine):
    m.eval()

n_params = lambda m: sum(p.numel() for p in m.parameters())
print("Tiny NeRF config:    ", TINY_CFG)
print("Extended NeRF config:", EXT_CFG)

## 2. Quantitative evaluation on held-out test views
**PSNR** $= -10\log_{10}(\mathrm{MSE})$ on $[0,1]$ images. **SSIM** uses an $11\times11$ Gaussian window ($\sigma=1.5$),
computed per channel and averaged (implemented below in PyTorch so no extra packages are needed).
For the extended model we report both the coarse-network output and the final fine output.

In [ ]:
def psnr(img, gt):
    return (-10. * torch.log10(F.mse_loss(img, gt))).item()

def ssim(img, gt, window_size=11, sigma=1.5, C1=0.01**2, C2=0.03**2):
    """SSIM for [H, W, 3] images in [0, 1]."""
    x = img.permute(2, 0, 1).unsqueeze(0); y = gt.permute(2, 0, 1).unsqueeze(0)
    coords = torch.arange(window_size, dtype=torch.float32, device=img.device) - window_size // 2
    g = torch.exp(-coords ** 2 / (2 * sigma ** 2)); g = g / g.sum()
    win = (g[:, None] @ g[None, :]).expand(3, 1, window_size, window_size).contiguous()
    conv = lambda t: F.conv2d(t, win, groups=3)        # 'valid' convolution
    mu_x, mu_y = conv(x), conv(y)
    s_xx = conv(x * x) - mu_x ** 2
    s_yy = conv(y * y) - mu_y ** 2
    s_xy = conv(x * y) - mu_x * mu_y
    s = ((2 * mu_x * mu_y + C1) * (2 * s_xy + C2)) / ((mu_x ** 2 + mu_y ** 2 + C1) * (s_xx + s_yy + C2))
    return s.mean().item()

renders = {'tiny': [], 'ext_coarse': [], 'ext': [], 'depth_tiny': [], 'depth_ext': []}
metrics = {k: {'psnr': [], 'ssim': []} for k in ('tiny', 'ext_coarse', 'ext')}
times = {'tiny': [], 'ext': []}

def timed(fn):
    if device.type == 'cuda': torch.cuda.synchronize()
    t0 = time.time(); out = fn()
    if device.type == 'cuda': torch.cuda.synchronize()
    return out, time.time() - t0

for i in range(len(test_images)):
    gt = test_images[i]
    rays_o, rays_d = get_rays(K, test_poses[i], H, W)
    o_t, dt_t = timed(lambda: render_full('tiny', rays_o, rays_d))
    o_e, dt_e = timed(lambda: render_full('ext', rays_o, rays_d))
    times['tiny'].append(dt_t); times['ext'].append(dt_e)
    for key, img in (('tiny', o_t['rgb']), ('ext_coarse', o_e['rgb_coarse']), ('ext', o_e['rgb'])):
        img = img.clamp(0, 1)
        renders[key].append(img)
        metrics[key]['psnr'].append(psnr(img, gt)); metrics[key]['ssim'].append(ssim(img, gt))
    renders['depth_tiny'].append(o_t['depth']); renders['depth_ext'].append(o_e['depth'])

names = {'tiny': 'Tiny NeRF (baseline)', 'ext_coarse': 'Extended - coarse net', 'ext': 'Extended - fine net'}
print(f"{'view':<6}" + "".join(f"{names[k]:>26}" for k in names))
for i in range(len(test_images)):
    print(f"{i:<6}" + "".join(f"{metrics[k]['psnr'][i]:>15.2f} / {metrics[k]['ssim'][i]:.3f}" for k in names))
print("-" * 84)
print(f"{'mean':<6}" + "".join(f"{np.mean(metrics[k]['psnr']):>15.2f} / {np.mean(metrics[k]['ssim']):.3f}" for k in names))
print("(PSNR dB / SSIM)")

In [ ]:
# Summary table for the report: quality, model size and cost
evals_per_ray = {'tiny': TINY_CFG['N_samples'], 'ext': EXT_CFG['N_c'] + (EXT_CFG['N_c'] + EXT_CFG['N_f'])}
rows = [
    ('Tiny NeRF', 'tiny', n_params(model_tiny), evals_per_ray['tiny'], np.mean(times['tiny'])),
    ('Extended NeRF', 'ext', n_params(model_coarse) + n_params(model_fine), evals_per_ray['ext'], np.mean(times['ext'])),
]
print(f"{'model':<16}{'PSNR (dB)':>12}{'SSIM':>8}{'params':>10}{'MLP evals/ray':>15}{'s / image':>11}")
for name, key, p, ev, t in rows:
    print(f"{name:<16}{np.mean(metrics[key]['psnr']):>12.2f}{np.mean(metrics[key]['ssim']):>8.3f}{p:>10,}{ev:>15}{t:>11.3f}")
gain = np.mean(metrics['ext']['psnr']) - np.mean(metrics['tiny']['psnr'])
print(f"\nPSNR gain of the extended model: {gain:+.2f} dB  (reference target: 28 dB average)")
print(f"Gain from the fine pass over the coarse pass: "
      f"{np.mean(metrics['ext']['psnr']) - np.mean(metrics['ext_coarse']['psnr']):+.2f} dB")

with open(os.path.join('results', 'metrics_summary.json'), 'w') as fh:
    json.dump({k: {'psnr_per_view': metrics[k]['psnr'], 'ssim_per_view': metrics[k]['ssim'],
                   'psnr_mean': float(np.mean(metrics[k]['psnr'])), 'ssim_mean': float(np.mean(metrics[k]['ssim']))}
               for k in metrics}, fh, indent=1)

## 3. Training curves (from the saved training histories)

In [ ]:
hist_tiny = json.load(open('results/tiny_history.json'))
hist_ext = json.load(open('results/extended_history.json'))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, key, title in ((axes[0], 'test_psnrs', 'Test PSNR (held-out views)'), (axes[1], 'train_psnrs', 'Train PSNR (fine output)')):
    ax.plot(np.arange(1, len(hist_tiny[key]) + 1), hist_tiny[key], label='Tiny NeRF')
    ax.plot(np.arange(1, len(hist_ext[key]) + 1), hist_ext[key], label='Extended NeRF')
    ax.axhline(28, ls='--', c='grey', lw=1, label='28 dB reference')
    ax.set_xlabel('epoch'); ax.set_ylabel('PSNR (dB)'); ax.set_title(title); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); savefig(fig, 'training_curves.png'); plt.show()
print(f"mean epoch time  Tiny: {np.mean(hist_tiny['epoch_times']):.1f}s   Extended: {np.mean(hist_ext['epoch_times']):.1f}s")

## 4. Qualitative comparison
### Test-view mosaic with absolute-error maps

In [ ]:
n = len(test_images)
fig, axes = plt.subplots(5, n, figsize=(2.3 * n, 11.5))
axes = np.array(axes).reshape(5, n)
row_titles = ['Ground truth', 'Tiny NeRF', 'Extended NeRF', '|error| Tiny', '|error| Extended']
for i in range(n):
    gt = test_images[i]
    err_t = (renders['tiny'][i] - gt).abs().mean(-1).cpu().numpy()
    err_e = (renders['ext'][i] - gt).abs().mean(-1).cpu().numpy()
    axes[0, i].imshow(gt.cpu().numpy()); axes[0, i].set_title(f'Test {i}')
    axes[1, i].imshow(renders['tiny'][i].cpu().numpy()); axes[1, i].set_title(f"{metrics['tiny']['psnr'][i]:.2f} dB", fontsize=9)
    axes[2, i].imshow(renders['ext'][i].cpu().numpy()); axes[2, i].set_title(f"{metrics['ext']['psnr'][i]:.2f} dB", fontsize=9)
    axes[3, i].imshow(err_t, cmap='inferno', vmin=0, vmax=0.3)
    axes[4, i].imshow(err_e, cmap='inferno', vmin=0, vmax=0.3)
for r in range(5):
    for c in range(n):
        axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
    axes[r, 0].set_ylabel(row_titles[r])
plt.tight_layout(); savefig(fig, 'test_mosaic.png'); plt.show()

### Zoomed crops (same region of the best-differing test view)

In [ ]:
# choose the test view where the two models differ most, and crop around the largest error reduction
diffs = [metrics['ext']['psnr'][i] - metrics['tiny']['psnr'][i] for i in range(n)]
v = int(np.argmax(diffs))
gain_map = ((renders['tiny'][v] - test_images[v]).abs() - (renders['ext'][v] - test_images[v]).abs()).mean(-1)
gain_map = F.avg_pool2d(gain_map[None, None], 15, stride=1, padding=7)[0, 0]
cy, cx = np.unravel_index(int(torch.argmax(gain_map)), gain_map.shape)
s = 20
y0, x0 = int(np.clip(cy - s, 0, H - 2 * s)), int(np.clip(cx - s, 0, W - 2 * s))
crop = lambda im: im[y0:y0 + 2 * s, x0:x0 + 2 * s].cpu().numpy()

fig, axes = plt.subplots(1, 4, figsize=(11, 3))
axes[0].imshow(test_images[v].cpu().numpy())
axes[0].add_patch(plt.Rectangle((x0, y0), 2 * s, 2 * s, fill=False, ec='red', lw=2)); axes[0].set_title(f'Test {v} (crop region)')
for ax, im, t in zip(axes[1:], (test_images[v], renders['tiny'][v], renders['ext'][v]), ('Ground truth', 'Tiny NeRF', 'Extended NeRF')):
    ax.imshow(crop(im), interpolation='nearest'); ax.set_title(t)
for ax in axes: ax.axis('off')
plt.tight_layout(); savefig(fig, 'zoom_crops.png'); plt.show()

### Expected depth maps $\hat{z} = \sum_i w_i t_i$ (geometry learned by each model)

In [ ]:
idx = list(range(min(3, n)))
fig, axes = plt.subplots(2, len(idx), figsize=(3 * len(idx), 6))
axes = np.array(axes).reshape(2, len(idx))
for j, i in enumerate(idx):
    for r, key, t in ((0, 'depth_tiny', 'Tiny NeRF'), (1, 'depth_ext', 'Extended NeRF')):
        im = axes[r, j].imshow(renders[key][i].cpu().numpy(), cmap='viridis', vmin=NEAR, vmax=FAR)
        axes[r, j].set_title(f'{t} - test {i}', fontsize=9); axes[r, j].axis('off')
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.7, label='depth')
savefig(fig, 'depth_maps.png'); plt.show()

### Novel views on a camera orbit (poses not in the dataset)

In [ ]:
trans_t = lambda t: torch.tensor([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, t], [0, 0, 0, 1]], dtype=torch.float32)
rot_phi = lambda phi: torch.tensor([[1, 0, 0, 0], [0, np.cos(phi), -np.sin(phi), 0],
                                    [0, np.sin(phi), np.cos(phi), 0], [0, 0, 0, 1]], dtype=torch.float32)
rot_theta = lambda th: torch.tensor([[np.cos(th), 0, -np.sin(th), 0], [0, 1, 0, 0],
                                     [np.sin(th), 0, np.cos(th), 0], [0, 0, 0, 1]], dtype=torch.float32)

def pose_spherical(theta, phi, radius):
    c2w = trans_t(radius)
    c2w = rot_phi(phi / 180. * np.pi) @ c2w
    c2w = rot_theta(theta / 180. * np.pi) @ c2w
    c2w = torch.tensor([[-1, 0, 0, 0], [0, 0, 1, 0], [0, 1, 0, 0], [0, 0, 0, 1]], dtype=torch.float32) @ c2w
    return c2w.to(device)

thetas = [0, 60, 120, 180, 240, 300]
fig, axes = plt.subplots(2, len(thetas), figsize=(2.3 * len(thetas), 5))
for j, th in enumerate(thetas):
    ro, rd = get_rays(K, pose_spherical(th, -30., 4.0), H, W)
    axes[0, j].imshow(render_full('tiny', ro, rd)['rgb'].clamp(0, 1).cpu().numpy())
    axes[1, j].imshow(render_full('ext', ro, rd)['rgb'].clamp(0, 1).cpu().numpy())
    axes[0, j].set_title(f'theta={th}', fontsize=9)
for r, t in enumerate(('Tiny NeRF', 'Extended NeRF')):
    for ax in axes[r]: ax.set_xticks([]); ax.set_yticks([])
    axes[r, 0].set_ylabel(t)
plt.tight_layout(); savefig(fig, 'novel_views.png'); plt.show()

## 5. Hierarchical sampling: where do the samples go?
For one ray that hits the object we plot the coarse weights $w_i = T_i\alpha_i$ (which define the PDF), the
positions of the $N_c$ stratified samples and the $N_c+N_f$ samples used by the fine network.
We also measure, over all test rays, the fraction of samples that fall where the fine weights are non-negligible.

In [ ]:
CHUNK = 1024   # rays per forward pass - keeps GPU memory low (raise it if you have a bigger GPU)

def chunked_extras(fn, ro, rd, chunk=CHUNK):
    """Run a render function over rays in chunks and concatenate its 'extras' dict."""
    parts = []
    for i in range(0, ro.shape[0], chunk):
        ex_i = fn(ro[i:i+chunk], rd[i:i+chunk])
        parts.append({k: v.detach() for k, v in ex_i.items()})
    return {k: torch.cat([p[k] for p in parts]) for k in parts[0]}

if device.type == 'cuda':
    torch.cuda.empty_cache()
with torch.no_grad():
    ro, rd = get_rays(K, test_poses[0], H, W)
    ro, rd = ro.reshape(-1, 3), rd.reshape(-1, 3)
    ex = chunked_extras(lambda o, d: render_rays(model_coarse, model_fine, o, d, NEAR, FAR, EXT_CFG['N_c'],
                                                 EXT_CFG['N_f'], rand=False, return_extras=True)[2], ro, rd)
    ex_t = chunked_extras(lambda o, d: render_rays_uniform(model_tiny, o, d, NEAR, FAR, TINY_CFG['N_samples'],
                                                           rand=False, return_extras=True)[1], ro, rd)

# pick the ray (pixel) with the highest opacity near the image centre
acc = ex['acc_fine'].reshape(H, W).clone()
mask = torch.zeros_like(acc); mask[H//4:3*H//4, W//4:3*W//4] = 1
r = int(torch.argmax(acc * mask))
tc, wc = ex['t_coarse'][r].cpu().numpy(), ex['weights_coarse'][r].cpu().numpy()
tf, wf = ex['t_fine'][r].cpu().numpy(), ex['weights_fine'][r].cpu().numpy()

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(tc, wc, 'o-', ms=3, label=f'coarse weights ({len(tc)} stratified samples)')
ax.plot(tf, wf, '.-', ms=3, alpha=0.8, label=f'fine weights ({len(tf)} samples)')
ax.plot(tc, np.full_like(tc, -0.06), '|', ms=10, c='C0')
ax.plot(tf, np.full_like(tf, -0.12), '|', ms=10, c='C1')
ax.set_xlabel('depth t along the ray'); ax.set_ylabel('weight $T_i\\alpha_i$'); ax.set_xlim(NEAR, FAR)
ax.set_title(f'Sample placement on one ray (pixel {r // W},{r % W} of test view 0); ticks = sample positions')
ax.legend(loc='upper right'); ax.grid(alpha=0.3)
plt.tight_layout(); savefig(fig, 'hierarchical_sampling_ray.png'); plt.show()

# How many samples land in the "useful" band of each ray? (samples within the depth interval that holds 98%
# of the final fine weight, evaluated on rays that actually hit the object)
def useful_fraction(t_samples, t_ref, w_ref, hit):
    cdf = torch.cumsum(w_ref, -1) / (w_ref.sum(-1, keepdim=True) + 1e-10)
    lo = torch.gather(t_ref, -1, torch.searchsorted(cdf, torch.full_like(cdf[:, :1], 0.01)).clamp(max=t_ref.shape[-1]-1))
    hi = torch.gather(t_ref, -1, torch.searchsorted(cdf, torch.full_like(cdf[:, :1], 0.99)).clamp(max=t_ref.shape[-1]-1))
    inside = ((t_samples >= lo) & (t_samples <= hi)).float().mean(-1)
    return inside[hit].mean().item()

hit = ex['acc_fine'] > 0.5
print(f"rays hitting the object: {hit.float().mean().item()*100:.1f}%")
print(f"fraction of samples inside the surface band   uniform ({TINY_CFG['N_samples']} pts, Tiny): "
      f"{useful_fraction(ex_t['t'], ex['t_fine'], ex['weights_fine'], hit)*100:.1f}%")
print(f"                                              coarse ({EXT_CFG['N_c']} pts):        "
      f"{useful_fraction(ex['t_coarse'], ex['t_fine'], ex['weights_fine'], hit)*100:.1f}%")
print(f"                                              fine   ({EXT_CFG['N_c'] + EXT_CFG['N_f']} pts):        "
      f"{useful_fraction(ex['t_fine'], ex['t_fine'], ex['weights_fine'], hit)*100:.1f}%")

## 6. View-dependent appearance
**(a)** We take a 3D surface point (the expected-depth point of a pixel) and query each model's colour while the viewing
direction sweeps a full circle around it. The Tiny NeRF ignores $\mathbf{d}$, so its curve is flat by construction;
any variation in the extended model is learned view-dependent radiance.
**(b)** We re-render a test view with the viewing direction of *every* ray replaced by one fixed direction. Density is
unchanged (it does not depend on $\mathbf{d}$), so the difference image isolates the view-dependent colour component.

In [ ]:
class FixedDirection(nn.Module):
    """Wraps a radiance field so every query uses the same viewing direction d0."""
    def __init__(self, net, d0):
        super().__init__(); self.net, self.d0 = net, d0
    def forward(self, x, dir):
        return self.net(x, self.d0.expand_as(dir))

# (a) colour of one surface point vs viewing azimuth -------------------------------------------
x_surf = (ro[r] + rd[r] * ex['depth_fine'][r]).reshape(1, 3)
az = torch.linspace(0, 2 * np.pi, 181, device=device)
el = np.deg2rad(-30.)
dirs = torch.stack([torch.cos(az) * np.cos(el), torch.sin(az) * np.cos(el), torch.full_like(az, np.sin(el))], -1)
with torch.no_grad():
    c_ext = model_fine(x_surf.expand_as(dirs), dirs)[..., 1:].cpu().numpy()
    c_tiny = model_tiny(x_surf.expand_as(dirs), dirs)[..., 1:].cpu().numpy()

# (b) fixed-direction re-render ------------------------------------------------------------------
v = 0
ro_v, rd_v = get_rays(K, test_poses[v], H, W)
d0 = -train_poses[0][:3, 2]                       # viewing direction of training camera 0
d0 = (d0 / d0.norm()).reshape(1, 3)
normal = render_full('ext', ro_v, rd_v)['rgb'].clamp(0, 1)
fixed = render_full('ext', ro_v, rd_v, coarse=FixedDirection(model_coarse, d0),
                    fine=FixedDirection(model_fine, d0))['rgb'].clamp(0, 1)
vd = (normal - fixed).abs().mean(-1)

fig = plt.figure(figsize=(13, 3.4))
ax = fig.add_subplot(1, 4, 1)
for ch, col in enumerate(('r', 'g', 'b')):
    ax.plot(np.rad2deg(az.cpu().numpy()), c_ext[:, ch], c=col, label=f'Extended {col.upper()}')
    ax.plot(np.rad2deg(az.cpu().numpy()), c_tiny[:, ch], c=col, ls='--', alpha=0.6)
ax.set_xlabel('view azimuth (deg)'); ax.set_ylabel('predicted colour'); ax.set_title('(a) colour vs view (-- = Tiny)')
ax.grid(alpha=0.3); ax.legend(fontsize=7)
for k, (im, t) in enumerate(((normal, f'(b) Extended, true dirs'), (fixed, 'same, fixed direction d0'))):
    a = fig.add_subplot(1, 4, k + 2); a.imshow(im.cpu().numpy()); a.set_title(t, fontsize=10); a.axis('off')
a = fig.add_subplot(1, 4, 4); m = a.imshow(vd.cpu().numpy(), cmap='magma'); a.set_title('|difference| (view-dependent part)', fontsize=10); a.axis('off')
fig.colorbar(m, ax=a, shrink=0.8)
plt.tight_layout(); savefig(fig, 'view_dependence.png'); plt.show()

print(f"colour range over the sweep - Extended: {np.ptp(c_ext, axis=0).round(3)}   Tiny: {np.ptp(c_tiny, axis=0).round(3)}")
print(f"mean |difference| from fixing the view direction: {vd.mean().item():.4f}  (max {vd.max().item():.3f})")
print(f"PSNR of fixed-direction render vs GT: {psnr(fixed, test_images[v]):.2f} dB  (true directions: {psnr(normal, test_images[v]):.2f} dB)")

All figures used in the report are saved in `results/figures/`.

In [ ]:
print(sorted(os.listdir(FIG_DIR)))